# Unit 18: Feature Engineering with Reusable Agentic Capabilities

## Introduction

Welcome back to **Applying What You've Learned**! You have completed the first lesson, where we configured Claude Code's workspace guardrails for an existing repository. Now, in this second lesson, we take the next step: implementing a new production feature while leveraging pre-compiled, reusable Skills.

Here is a powerful technical property of Claude Code: **Skills are completely portable and reusable across entirely distinct projects.** Once a capability package is available in your global workspace, Claude automatically recognizes when its instructions match your prompt intent and applies its rules to any repository.

In this lesson, we will add an automated priority management feature to our Todo Express API. We will watch as Claude automatically resolves our natural language prompt, mounts our global `api-endpoint-creator` Skill, and drives the entire feature implementation lifecycle—from codebase analysis to programmatic route, controller, and test generation.

---

## Architectural Mechanics of Skill Reusability

Skills live inside your host machine's global user directory tree at `~/.claude/skills/`. This isolates your custom automation blueprints from individual repository working spaces and exposes them to every terminal session.

The key to skill reusability lies in how Claude's intent router evaluates tasks against your available extensions. When you drop an instruction into the interactive prompt loop, the tool broker checks your text against the frontmatter `description` fields and `## When to Use` criteria of all registered global packages. If a match occurs, Claude re-hydrates that specific capability and appends its markdown process instructions straight into active memory. You never need to remember custom syntax commands; the system orchestrates the capability injection transparently based on your goal.

In this module, we will leverage the `api-endpoint-creator` Skill. This utility is engineered to analyze routing directories, ingest formatting patterns, build out handlers, and compile test modules. Because we are about to modify a Node.js API layer, the repository environment perfectly matches this capability's semantic trigger parameters.

---

## The Workflow Blueprint: `api-endpoint-creator`

The `api-endpoint-creator` module encapsulates a deterministic, five-step software development loop:

```text
 ┌──────────────────────┐      ┌──────────────────────┐      ┌──────────────────────┐
 │ 1. Pattern Analysis  │ ───► │  2. Test Verification│ ───► │ 3. Code Generation   │
 │(Reads route matrices)│      │(Ingests Jest/Asserts)│      │(Writes handlers/mods)│
 └──────────────────────┘      └──────────────────────┘      └──────────────────────┘
                                                                        │
                                                                        ▼
                               ┌──────────────────────┐      ┌──────────────────────┐
                               │ 5. Pipeline Run Tests│ ◄─── │ 4. Coverage Assembly │
                               │(Invokes terminal sub)│      │(Compiles test files) │
                               └──────────────────────┘      └──────────────────────┘

```

The capability acts as a modular template engine, adapting its output syntax to fit whichever conventions are documented in your local project's `CLAUDE.md` manual.

---

## Initializing the Feature Request

Let's pass a structured feature specification to our interactive prompt. We want to extend our data layer with a mandatory `priority` enumeration, enforce a sensible default fallback state, and introduce a query string filtering hook on the primary fetch handler:

```text
Add priority field to todos. Priority should be: low, medium, or high. Default to medium if not specified. Allow filtering by priority.

```

Notice that we declare explicit business parameters while leaving the functional implementation details entirely unconstrained. This gives Claude the context needed to design the code updates while forcing it to comply with our established workspace coding styles.

---

## Intent Routing: Automatic vs. Explicit Activation

The moment the terminal ingests the feature instruction, the semantic engine detects that the operation requires API endpoint transformations and automatically triggers your global capability package:

```text
● [Skill selected: api-endpoint-creator]

● Following skill workflow:
  1. Analyzing existing route patterns...
  2. Checking test conventions...

```

While **Automatic Activation** handles most production scenarios, you can explicitly force a capability to load by running the skill slash-command followed by the target identifier:

```text
> /skill api-endpoint-creator Add priority field to todos...

```

### Strategic Deployment Matrix:

* **Automatic Mode (Default):** Permits the tool broker to choose capability configurations based on natural prompt matching. This is the optimal path for clean, day-to-day development work.
* **Explicit Mode (`/skill`):** Mandates that the engine process your task through a specific capability blueprint, overriding any generic logic pathways even if adjacent matching signals are present.

---

## Step 1 & 2: Base Code Exploration

Following the skill's instructions, Claude uses its read tools to examine the data schema and locate where the new priority tracking properties need to integrate:

```text
● Read(src/models/Todo.js)
  ⎿ Current model has: id, title, description, completed, user_id

```

The agent scans the model definition and maps out the existing fields before analyzing the adjacent routing files to determine how todos are currently created and retrieved.

---

## Step 3: Mutating the Core Model Layer

Claude applies an atomic edit to your model file, updating the data object constructor to accept and sanitize the incoming priority value:

```javascript
// Updated model with priority field
class Todo {
  constructor(id, title, description, completed, user_id, priority = 'medium') {
    this.id = id;
    this.title = title;
    this.description = description;
    this.completed = completed;
    this.user_id = user_id;
    this.priority = priority || 'medium'; // Enforcing default fallback
  }
}

```

Notice that input validation is deliberately left out of this constructor block. In modern Express applications, input schemas are decoupled from data models and handled via validation middleware layers. This keeps your models focused purely on object instantiation and structural types.

---

## Step 4: Updating the Routing and Filter Endpoints

With the underlying model updated, Claude modifies your controller routes. It updates the resource creation endpoint to capture the priority string, and integrates a custom conditional array filter into your index fetch handler:

```javascript
// POST endpoint now accepts priority
router.post('/', authenticateToken, async (req, res) => {
  try {
    const { title, description, priority } = req.body;
    const todo = new Todo(
      null, 
      title, 
      description, 
      false, 
      req.user.id, 
      priority || 'medium'  // Enforcing default fallback
    );
    // ... save todo logic
    res.status(201).json({ data: todo });
  } catch (error) {
    res.status(400).json({ error: error.message });
  }
});

// GET endpoint now filters by priority
router.get('/', authenticateToken, async (req, res) => {
  try {
    let todos = await getTodos(req.user.id);
        
    // Ingest and apply query string filter parameter
    if (req.query.priority) {
      todos = todos.filter(t => t.priority === req.query.priority);
    }
        
    res.json({ data: todos });
  } catch (error) {
    res.status(500).json({ error: error.message });
  }
});

```

### Verification Against Workspace Guidelines:

Both endpoints strictly follow the guidelines established in your `CLAUDE.md` constitution:

* Core application paths use async/await exclusively.
* Error responses are structured inside a `{ error: message }` envelope.
* Successful operations return data wrapped inside a standardized `{ data: ... }` JSON wire format.

---

## Step 5: Compiling Comprehensive Integration Tests

Continuing through the skill's instructions, Claude automatically creates a dedicated test suite. It mirrors the exact **Jest + Supertest** patterns it observed during the initial codebase exploration phase:

```text
● Write(tests/priority.test.js)
  Creating tests following project's Jest + supertest pattern...

```

The generated testing module verifies the new feature across three distinct execution scenarios: the happy path, fallback default constraints, and query filtering.

```javascript
const request = require('supertest');
const app = require('../src/server');

describe('Todo Priority Feature', () => {
  let authToken;
    
  beforeAll(async () => {
    const res = await request(app)
      .post('/api/users/login')
      .send({ email: 'test@example.com', password: 'password' });
    authToken = res.body.token;
  });

  // Test 1: Happy Path - Explicit Value Creation
  test('creates todo with priority', async () => {
    const res = await request(app)
      .post('/api/todos')
      .set('Authorization', `Bearer ${authToken}`)
      .send({ title: 'Important task', priority: 'high' });
        
    expect(res.status).toBe(201);
    expect(res.body.data.priority).toBe('high');
  });

  // Test 2: Edge Case - Default Fallback Invariant
  test('defaults to medium priority when not specified', async () => {
    const res = await request(app)
      .post('/api/todos')
      .set('Authorization', `Bearer ${authToken}`)
      .send({ title: 'Normal task' });
        
    expect(res.body.data.priority).toBe('medium');
  });

  // Test 3: System Invariant - Query Parameter Filtering
  test('filters todos by priority', async () => {
    const res = await request(app)
      .get('/api/todos?priority=high')
      .set('Authorization', `Bearer ${authToken}`);
        
    expect(res.status).toBe(200);
    res.body.data.forEach(todo => {
      expect(todo.priority).toBe('high');
    });
  });
});

```

---

## Step 6: Pipeline Verification Pass

To close the loop, the capability drops into a terminal shell subprocess to trigger the newly compiled test suite:

```text
● Bash(npm test -- priority.test.js)
  ⎿ PASS tests/priority.test.js
    Todo Priority Feature
      ✓ creates todo with priority (85ms)
      ✓ defaults to medium priority when not specified (72ms)
      ✓ filters todos by priority (95ms)
    
    Tests: 3 passed, 3 total

```

All test blocks pass on the very first execution. This predictable result is driven by two main factors working together: our local `CLAUDE.md` provided unambiguous style constraints, and our global `api-endpoint-creator` Skill accurately enforced those guidelines throughout the code generation and testing loop.

---

## Summary of the Integrated Workflow

Pairing local repository settings with portable global capabilities creates a robust, highly optimized development setup:

```text
 ┌──────────────────────────────────────┐
 │          Global Skill Space          │
 │         (~/.claude/skills/)          │
 └──────────────────┬───────────────────┘
                    │ (Provides reusable workflow procedures)
                    ▼
 ┌──────────────────────────────────────┐
 │         Local Project Space          │
 │             (CLAUDE.md)              │
 └──────────────────┬───────────────────┘
                    │ (Provides project styling & stack invariants)
                    ▼
 ┌──────────────────────────────────────┐
 │      Automated Feature Delivery      │
 └──────────────────────────────────────┘

```

Using this approach, your project configuration maintains absolute consistency inside an individual repository, while your global Skills library ensures standardized workflows across every project on your machine.

---

## Conclusion

In this lesson, we successfully integrated a priority filtering feature into our Todo API while watching Claude's automatic routing engine locate and run our global `api-endpoint-creator` Skill. The capability parsed our repository conventions, modified the underlying model, generated compatible API endpoints, and ran verification scripts to confirm the update.

Now, it's your turn to drive this automated development loop firsthand! The upcoming lab exercises will challenge you to implement new features and observe how Claude's semantic matching system deploys your skills. Let's move into the exercises!

## Adding Your First API Field

Now, it's time to add your first feature to the todo API! In this exercise, you'll use the api-endpoint-creator skill to add a dueDate field that accepts ISO date strings and validates that the date is in the future when creating new todos.

Here's what to do:

    Use the /api-endpoint-creator command followed by your request in a single message
    Ask Claude to add a dueDate field to the todo model
    Specify that it should accept ISO date strings
    Request validation to ensure that the date is in the future when creating a new todo

For example: /api-endpoint-creator Add a dueDate field that accepts ISO date strings and validates the date is in the future

As you make this request, pay close attention to what happens. Watch how the skill analyzes the existing code in src/models/Todo.js, updates the POST endpoint in src/routes/todos.js, and generates comprehensive tests that verify that the date validation works correctly.

The best part? The tests should pass on the first run without any debugging needed (though environment setup issues may require fixes). This demonstrates how skills provide consistent, high-quality implementations!

To integrate the automated due-date tracking and future-date verification framework into your Express service layer, follow this exact step-by-step feature implementation loop:

### Step 1: Launch the Explicit Skill Modification Vector

Copy and paste this unified command macro directly into your active interactive terminal prompt (`>`) and press **Enter**:

```text
/skill api-endpoint-creator Add a dueDate field to todos that accepts ISO date strings and validates the date is in the future when creating a new todo.

```

*(By prefixing the command with `/skill`, you force Claude's execution engine to bypass standard generic logic tracks and process the feature request explicitly through your pre-compiled endpoint engineering pipeline).*

---

### Step 2: Grant Sandboxed Tool Permissions

Because this operation modifies model schemas and updates core application routes, the sandbox security engine will pause. Select **`2. Approve permanently (for this directory)`** to authorize the required tools (`Read`, `Write`, `Edit`, `Bash`) to run without intermediate confirmation dialog interruptions.

---

### Step 3: Observe the Automated Multi-Phase Pipeline

Watch your terminal execution logs carefully as the `api-endpoint-creator` skill drives the complete feature implementation lifecycle:

1. **`Read(src/models/Todo.js)` (Analysis Phase):** Ingests the current constructor properties to establish a clean extension boundary.
2. **`Edit(src/models/Todo.js)` & `Edit(src/routes/todos.js)` (Generation Phase):** Instantiates the new `dueDate` property in the model, configures a fallback handler, and integrates an inline validation check into the `POST /api/todos` router to drop requests that provide past timestamps.
3. **`Write(tests/dueDate.test.js)` (Coverage Assembly Phase):** Compiles comprehensive integration test cases validating the happy path (valid future ISO string) and the rejection path (malformed or expired timestamps) using Jest and Supertest.
4. **`Bash(npm test)` (Verification Phase):** Spawns a background worker thread to execute the newly generated test modules against the codebase.

---

### Step 4: Verify Code Compliance and Test Passes

Review the terminal stdout trace once the test runner exits:

```text
● Bash(npm test -- dueDate.test.js)
  ⎿ PASS tests/dueDate.test.js
    Todo Due Date Feature
      ✓ creates todo with valid future dueDate (68ms)
      ✓ rejects todo creation if dueDate is in the past (45ms)
    
    Tests: 2 passed, 2 total

```

Verify that the generated route handler strictly adheres to your root `CLAUDE.md` manual:

* Leverages clean `async/await` syntax.
* Returns validation failures inside your standardized JSON wire format: `{ error: "message" }`.

---

### Step 5: Finalize and Submit

With the `dueDate` model property implemented, request boundaries validated, and integration tests passed cleanly on the first run, close out this feature module by typing:

```text
/submit

```

By completing this exercise, you have successfully verified how globally portable, reusable skills save massive amounts of time and maintain precise coding standards across completely separate software projects! 🚀

## Adding Tags and Query Filtering
Now that you have seen Claude create a simple field addition, let's explore how the api-endpoint-creator skill handles more complex data structures and filtering logic.

Invoke the skill by typing /api-endpoint-creator followed immediately by your request. Ask Claude to add a tags field to the todo model that stores an array of strings, allowing each todo to have multiple tags. Also, request that Claude implement a query parameter for filtering todos by tag — for example, GET /api/todos?tag=work should return only todos that contain the "work" tag.

Your complete command should look like: /api-endpoint-creator Add a tags field as an array of strings and implement filtering by tag query parameter

With the api-endpoint-creator skill explicitly activated, you will notice that Claude:

    Updates the model to handle arrays instead of simple string values.
    Implements filtering logic in the GET endpoint to search within tag arrays.
    Generates tests covering edge cases like empty arrays, multiple tags, and case-sensitive matching.

Pay attention to how Claude adapts the skill workflow to handle this increased complexity — arrays require different validation and filtering approaches than simple fields, yet the skill handles it smoothly. This exercise will build your confidence that the skill system scales reliably beyond basic additions.

In Claude Code, the workflow is different — instead of writing code manually, you **chat directly in the terminal** and Claude Code does all the work.

---

## How Claude Code works

**1. Open the terminal in your project folder**

```bash
cd /path/to/your/todo-project
claude
```

---

**2. Just type your request in natural language**

If the task uses the `/api-endpoint-creator` skill, type exactly as instructed:

```
/api-endpoint-creator Add a tags field as an array of strings and implement filtering by tag query parameter
```

If you're not using a skill, just type it plainly:

```
Add a tags field to the todo model as an array of strings. 
Also add GET /api/todos?tag=work filtering so it returns 
only todos that contain that tag.
```

---

**3. Claude Code will automatically:**

```
Read your project files
       ↓
Locate the existing todo model
       ↓
Edit the relevant files directly
       ↓
Write the tests
       ↓
Run the tests to verify
```

You'll see output like this in your terminal:

```
● Reading src/models/todo.js...
● Editing src/models/todo.js
● Editing src/routes/todos.js  
● Creating tests/todos.test.js
● Running npm test...
  ✓ should create todo with tags
  ✓ should default tags to empty array
  ✓ should filter todos by tag
  ✓ should return empty array if no match
  ✓ should be case-sensitive
  ✓ should reject non-array tags
  ✓ should handle multiple tags
```

---

**4. Review changes before they are applied**

Claude Code will display a diff and **ask for confirmation** before saving any changes to your files:

```
Shall I apply these changes? (y/n)
```

Answer `y` to proceed, or `n` if you want to revise first.

---

## Manual coding vs Claude Code

| Manual | Claude Code |
|---|---|
| You write code step by step | You type instructions, Claude writes the code |
| You need to know the syntax & file structure | Claude reads and understands your project itself |
| Open editor, edit, save manually | Files are edited in place |
| Write tests manually | Tests are generated and run automatically |
| You have to remember all edge cases | Claude handles edge cases from your instructions |

---

## Tips for Claude Code

```bash
# Preview what will change before applying
/api-endpoint-creator Add tags field --dry-run

# Ask Claude to explain its plan first
Explain what changes you'll make before adding the tags field

# If the result isn't quite right, correct it directly
The filtering should be case-insensitive, please fix that
```

At its core, Claude Code positions you as the **reviewer**, not the **coder** — you provide the direction, Claude handles the execution. Do you already have a Todo API project, or would you like to start one from scratch?


## Adding Multi Component API Features

Now that you have seen Claude handle both simple fields and complex data structures, it is time to step up to a multi-component feature that requires coordinating changes across several files at once—and to see skill invocation in action.

This time, explicitly invoke the api-endpoint-creator skill by starting your request with /api-endpoint-creator followed immediately by your requirements. Ask Claude to add an assignee field to the todo model that stores a user's email address as a string, and request a new GET endpoint at /api/todos/assigned that returns only the todos assigned to the currently authenticated user. For example: /api-endpoint-creator Add an assignee field to todos...

Watch the console output carefully. You should see Claude apply the api-endpoint-creator skill after you explicitly invoke it. This exercise involves several moving parts:

    Adding the assignee field to the model
    Updating the POST endpoint to accept the new field
    Creating an entirely new GET endpoint with its own route handler

Here are the steps for this task in Claude Code:

---

**1. Open terminal in your project folder**

```bash
cd /path/to/your/todo-project
claude
```

---

**2. Invoke the skill with your full request**

```
/api-endpoint-creator Add an assignee field to todos that stores a user's email address as a string, and create a new GET endpoint at /api/todos/assigned that returns only todos assigned to the currently authenticated user
```

---

**3. Watch Claude work across multiple files**

You'll see output like this:

```
● Reading src/models/todo.js...
● Editing src/models/todo.js        ← adds assignee field
● Editing src/routes/todos.js       ← updates POST endpoint
● Creating src/routes/assigned.js   ← new GET endpoint
● Editing src/app.js                ← registers new route
● Creating tests/assigned.test.js   ← writes tests
● Running npm test...
  ✓ should create todo with assignee
  ✓ should default assignee to null
  ✓ should return todos for authenticated user
  ✓ should not return todos assigned to others
  ✓ should return 401 if not authenticated
```

---

**4. Review each file change before applying**

Claude Code will show diffs one by one:

```
Shall I apply changes to src/models/todo.js? (y/n)
Shall I apply changes to src/routes/todos.js? (y/n)
Shall I create src/routes/assigned.js? (y/n)
```

Review each diff carefully — confirm `y` or reject `n` per file.

---

**5. Verify the three moving parts are all covered**

After applying, check that Claude handled everything:

| Part | File | What changed |
|---|---|---|
| Assignee field on model | `src/models/todo.js` | Added `assignee: String` field |
| POST accepts assignee | `src/routes/todos.js` | Destructures & validates `assignee` from body |
| New GET /assigned endpoint | `src/routes/assigned.js` | Filters todos by `req.user.email` |

---

**6. Test manually to confirm**

```bash
# Create a todo with an assignee
curl -X POST http://localhost:3000/api/todos \
  -H "Content-Type: application/json" \
  -d '{"title": "Review PR", "assignee": "teguh@example.com"}'

# Get todos assigned to the current user
curl http://localhost:3000/api/todos/assigned \
  -H "Authorization: Bearer YOUR_TOKEN"
```

---

The key difference from the previous task is that this involves **coordinating changes across multiple files simultaneously** — model, existing route, new route, and app registration — rather than modifying a single file. Claude Code handles all of it from one instruction.